<a href="https://colab.research.google.com/github/z0n6/universal-ai-transcriber/blob/main/Transcriber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Universal AI Transcriber (萬用音訊/影片轉錄神器)

這是一個基於 `faster-whisper` 開發的高效能語音轉錄工具。利用免費的 Google Colab GPU，一鍵將 Podcast、會議記錄或影片音訊轉換為高精準度的**逐字稿**與**專業字幕檔**。

### ✨ 特色亮點
* ⚡ **極速匯出**：轉錄與檔案生成分離，可秒速匯出不同格式，無需重複等待。
* 🎯 **繁中優化**：預設使用對繁簡轉換與台灣口音辨識度最佳的 `large-v2` 模型。
* 🛠️ **全格式支援**：支援易讀文件 (Docx/TXT) 與專業字幕 (SRT/VTT)。

> 💡 **使用說明**：請依序點擊下方三個步驟的「Play 按鈕 (▶️)」即可完成轉錄。

In [ ]:
# @title 🛠️ 第一步：初始化環境 (點擊左側 Play 按鈕)
# @markdown 系統將自動安裝 Whisper 模型、Docx 處理工具與音訊處理庫。
# @markdown <br><font size="2" color="gray">⏳ 初次執行約需 1-2 分鐘，請耐心等候...</font>

import os
import sys
import time
from datetime import datetime
from google.colab import files
from tqdm.notebook import tqdm

# 隱藏安裝過程的輸出，保持介面乾淨
from IPython.utils import io
import warnings
warnings.filterwarnings('ignore')

print("🔄 正在安裝必要套件...", end="")

with io.capture_output() as captured:
    !pip install -q faster-whisper python-docx yt-dlp

# 預先載入必要的 Library，避免在主程式才報錯
from faster_whisper import WhisperModel
from docx import Document

print()
print("✅ 安裝完成！請繼續執行下一步。")

In [ ]:
# @title 🎙️ 第二步：開始轉錄 (等待時間較長，但同一音檔僅需執行一次)

import json

# @markdown #### **1. 輸入設定**
podcast_url = "https://rss.soundon.fm/rssf/954689a5-3096-43a4-a80b-7810b219cef3/feedurl/5fbff599-9897-422a-812f-479eb0d75bd1/rssFileVip.mp3?timestamp=1769238608784" # @param {type:"string"}

# @markdown #### **2. 模型與轉錄設定**
model_size = "large-v2" # @param ["medium", "large-v2", "large-v3"]
# @markdown <font size="2" color="#0066cc"><b>💡 開發者建議：</b>繁體中文轉錄強烈建議維持 <b>large-v2</b>。經實測，v2 在台灣口音與繁簡轉換的穩定性上顯著優於 v3。</font>
initial_prompt = "\u7E41\u9AD4\u4E2D\u6587\u3002\u4EE5\u4E0B\u662F\u5C08\u6709\u540D\u8A5E\uFF1A\u764C\u5927, \u5B5F\u606D, \u8AFE\u4E9E, \u4E3B\u59D4\u3002\u8A71\u984C\u5305\u542B\u7F8E\u80A1\u3001\u53F0\u80A1\u8207\u6295\u8CC7\u5FC3\u6CD5\u3002\u958B\u5834\u767D\uFF1A\u6B61\u8FCE\u6536\u807D\u80A1\u764C\uFF0C\u6211\u662F\u8B1D\u5B5F\u606D\u3002" # @param {type:"string"}

def download_audio(url):
    print(f"⬇️ 正在下載音檔：{url} ...")
    output_filename = "podcast_audio.mp3"
    !wget -q -O {output_filename} {url}
    return output_filename

# 主執行邏輯
if podcast_url:
    try:
        audio_file = download_audio(podcast_url)

        print(f"🚀 載入模型 ({model_size})... 請稍候")
        model = WhisperModel(model_size, device="cuda", compute_type="float16")

        print("✍️ 開始轉錄... (請耐心等候，完成後資料將暫存)")
        segments, info = model.transcribe(audio_file, beam_size=5, initial_prompt=initial_prompt, language="zh")

        total_duration = info.duration
        pbar = tqdm(total=round(total_duration), unit="sec", desc="轉錄進度")

        # 將轉錄結果暫存為標準化字典列表
        raw_results = []
        for segment in segments:
            raw_results.append({
                "start": segment.start,
                "end": segment.end,
                "text": segment.text.strip()
            })
            pbar.n = min(round(segment.end), round(total_duration))
            pbar.refresh()
        pbar.close()

        # 儲存為 JSON，供下一步使用
        with open("transcription_raw.json", "w", encoding="utf-8") as f:
            json.dump(raw_results, f, ensure_ascii=False, indent=2)

        print("\n✅ 轉錄核心工作完成！原始資料已暫存。請至「第三步」選擇下載格式。")

    except Exception as e:
        print(f"❌ 發生錯誤: {e}")
else:
    print("⚠️ 請輸入 Podcast 網址。")

In [ ]:
# @title 📥 第三步：自訂檔名與下載 (秒速完成)
import json
import math
import re
from datetime import datetime
from google.colab import files
from docx import Document

# @markdown #### **1. 設定輸出檔名**
# @markdown <font size="2" color="gray">請輸入檔案名稱 (系統會自動補上副檔名)</font>
output_filename_base = "EP630" # @param {type:"string"}

# @markdown ---
# @markdown #### **2. 選擇需要的檔案格式 (可複選)**
export_srt = False # @param {type:"boolean"}
export_vtt = False # @param {type:"boolean"}
export_txt = False # @param {type:"boolean"}
export_docx = True # @param {type:"boolean"}

def format_timestamp(seconds, format_type="standard"):
    """轉換秒數為不同格式的時間戳"""
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)

    if format_type == "srt":
        millis = int((seconds - int(seconds)) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"
    elif format_type == "vtt":
        millis = int((seconds - int(seconds)) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:03d}"
    else: # docx, txt 使用的易讀格式
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def sanitize_filename(filename):
    """防呆機制：清理檔名中的不合法字元，若為空則用時間戳代入"""
    if not filename.strip():
        return f"transcript_{datetime.now().strftime('%Y%m%d_%H%M')}"
    # 移除 Windows/Linux 檔名不允許的字元 \ / : * ? " < > |
    return re.sub(r'[\\/:*?"<>|]', '_', filename)


# 檢查是否有轉錄資料
if not os.path.exists("transcription_raw.json"):
    print("⚠️ 找不到轉錄資料，請先執行「第二步」！")
else:
    # 讀取轉錄資料
    with open("transcription_raw.json", "r", encoding="utf-8") as f:
        segments = json.load(f)

    download_list = []

    # 處理檔名
    safe_base_name = sanitize_filename(output_filename_base)
    print(f"📂 準備產生檔案，基礎檔名為: {safe_base_name}")

    # 1. 產生 SRT (標準字幕)
    if export_srt:
        srt_name = f"{safe_base_name}.srt"
        with open(srt_name, "w", encoding="utf-8") as f:
            for i, seg in enumerate(segments, start=1):
                start = format_timestamp(seg['start'], "srt")
                end = format_timestamp(seg['end'], "srt")
                f.write(f"{i}\n{start} --> {end}\n{seg['text']}\n\n")
        download_list.append(srt_name)

    # 2. 產生 VTT (網頁影片常用字幕)
    if export_vtt:
        vtt_name = f"{safe_base_name}.vtt"
        with open(vtt_name, "w", encoding="utf-8") as f:
            f.write("WEBVTT\n\n")
            for seg in segments:
                start = format_timestamp(seg['start'], "vtt")
                end = format_timestamp(seg['end'], "vtt")
                f.write(f"{start} --> {end}\n{seg['text']}\n\n")
        download_list.append(vtt_name)

    # 3. 產生 DOCX (易讀文件)
    if export_docx:
        doc_name = f"{safe_base_name}.docx"
        doc = Document()
        doc.add_heading(f'轉錄內容 - {safe_base_name}', 0)
        for seg in segments:
            p = doc.add_paragraph()
            start = format_timestamp(seg['start'], "standard")
            p.add_run(f"[{start}] ").bold = True
            p.add_run(seg['text'])
        doc.save(doc_name)
        download_list.append(doc_name)

    # 4. 產生 TXT (純文字備份)
    if export_txt:
        txt_name = f"{safe_base_name}.txt"
        with open(txt_name, "w", encoding="utf-8") as f:
            for seg in segments:
                start = format_timestamp(seg['start'], "standard")
                f.write(f"[{start}] {seg['text']}\n")
        download_list.append(txt_name)

    # 觸發下載
    if download_list:
        print(f"✅ 完成！共產生 {len(download_list)} 個檔案，即將下載。")
        for file in download_list:
            files.download(file)
    else:
        print("⚠️ 未勾選任何輸出格式。")